In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
import os

In [2]:
sys.path.append(os.path.join(os.getcwd(), "..", "src"))

from data_loader import DataLoader
from feature_engineering import FeatureEngineering
from preprocessing import Preprocessing
from train import Trainer
from evaluate import Evaluator
from predict import Predictor

from sklearn.model_selection import train_test_split

### **Loading raw data**

In [3]:
loader = DataLoader(subfolder="raw")
loader.load("ev_market_2026.csv").validate().info()
df = loader.get_df()

2026-06-10 15:42:12 | INFO     | data_loader | DataLoader initialized | subfolder: raw
2026-06-10 15:42:12 | INFO     | data_loader | Loading file | path: c:\Users\mrcoo\OneDrive\Desktop\ev-price-prediction\data\raw\ev_market_2026.csv
2026-06-10 15:42:12 | INFO     | data_loader | file loaded successfully | shape: (2000, 24) | columns: ['brand', 'model', 'year', 'variant', 'price_usd', 'battery_capacity_kwh', 'range_miles', 'charging_speed_kw', 'acceleration_0_60_mph', 'top_speed_mph', 'horsepower', 'torque_nm', 'drive_type', 'seating_capacity', 'body_type', 'cargo_volume_cubic_ft', 'weight_kg', 'safety_rating', 'autopilot_level', 'country_of_origin', 'market_segment', 'annual_sales_units', 'customer_rating', 'warranty_years'] 
2026-06-10 15:42:12 | INFO     | data_loader | Running validation checks...
2026-06-10 15:42:12 | WARNING  | data_loader | Columns with missing values:
Series([], )
2026-06-10 15:42:12 | INFO     | data_loader | No duplicate rows found
2026-06-10 15:42:12 | INFO

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  2000 non-null   str    
 1   model                  2000 non-null   str    
 2   year                   2000 non-null   int64  
 3   variant                2000 non-null   str    
 4   price_usd              2000 non-null   float64
 5   battery_capacity_kwh   2000 non-null   float64
 6   range_miles            2000 non-null   float64
 7   charging_speed_kw      2000 non-null   float64
 8   acceleration_0_60_mph  2000 non-null   float64
 9   top_speed_mph          2000 non-null   float64
 10  horsepower             2000 non-null   float64
 11  torque_nm              2000 non-null   float64
 12  drive_type             2000 non-null   str    
 13  seating_capacity       2000 non-null   int64  
 14  body_type              2000 non-null   str    
 15  cargo_volume_cu

### **2. Split into train / test  (before ANY processing — no leakage)**

In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"\nSplit → train: {train_df.shape} | test: {test_df.shape}")


Split → train: (1600, 24) | test: (400, 24)


### **3. Feature engineering**

In [6]:
fe_train = FeatureEngineering(train_df)
train_df = (fe_train
            .add_efficiency_features()
            .add_performance_score()
            .add_age_feature()
            .add_value_score()
            .get_df())

fe_test = FeatureEngineering(test_df)
test_df = (fe_test
           .add_efficiency_features()
           .add_performance_score()
           .add_age_feature()
           .add_value_score()
           .get_df())

print(f"After feature engineering → train: {train_df.shape} | test: {test_df.shape}")


2026-06-10 15:56:16 | INFO     | feature_engineering | FeatureEngineering initialized | shape: (1600, 24)
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding efficiency features...
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding performance score...
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding vehicle age feature...
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding value score feature...
2026-06-10 15:56:16 | INFO     | feature_engineering | Returning feature-engineered dataframe | shape: (1600, 29)
2026-06-10 15:56:16 | INFO     | feature_engineering | New features added: ['range_per_kwh', 'charge_rate', 'performance_score', 'vehicle_age', 'value_score']
2026-06-10 15:56:16 | INFO     | feature_engineering | FeatureEngineering initialized | shape: (400, 24)
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding efficiency features...
2026-06-10 15:56:16 | INFO     | feature_engineering | Adding performance score...
2026-06-10

### **4. Preprocessing  (fit on train, transform test)**

In [5]:
preprocessor = Preprocessing(df)

preprocessed_df = (
    preprocessor
    .handling_missing_values()
    .encoding(target_col="price_usd")
    .scaling(target_col="price_usd")
    .get_df()
)


2026-06-04 23:39:39 | INFO     | preprocessing | Preprocessing initialized | shape (2000, 24)
2026-06-04 23:39:39 | INFO     | preprocessing | Handling missing values...
2026-06-04 23:39:39 | INFO     | preprocessing | Missing values handled successfully
2026-06-04 23:39:39 | INFO     | preprocessing | Encoding categorical columns...
2026-06-04 23:39:39 | INFO     | preprocessing | Encoding complete
2026-06-04 23:39:39 | INFO     | preprocessing | Scaling numerical columns...
2026-06-04 23:39:39 | INFO     | preprocessing | Scaling complete | scaled 32 columns
2026-06-04 23:39:39 | INFO     | preprocessing | Returning preprocessed dataframe | shape: (2000, 33)
